In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import os
import cv2
from skimage.feature import hog

# Image Read

## Train Dataset

In [2]:
dogs_path = r"C:\Users\KIIT0001\Downloads\Train_dataset_C&D\dogs"

In [3]:
cats_path = r"C:\Users\KIIT0001\Downloads\Train_dataset_C&D\cats"

In [4]:
data = []
labels = []

In [5]:
for img_name in os.listdir(dogs_path):
    img_path = os.path.join(dogs_path, img_name)
    
    img = cv2.imread(img_path)

    if img is None:          # IMPORTANT CHECK
        print("Skipping:", img_name)
        continue

    img = cv2.resize(img, (64, 64))
    img = img.flatten()

    data.append(img)
    labels.append(1)  # dog

Skipping: _DS_Store


In [6]:
for img_name in os.listdir(cats_path):
    img_path = os.path.join(cats_path, img_name)
    
    img = cv2.imread(img_path)

    if img is None:
        print("Skipping:", img_name)
        continue

    img = cv2.resize(img, (64, 64))
    img = img.flatten()

    data.append(img)
    labels.append(0)  # cat

Skipping: _DS_Store


In [7]:
X_train = np.array(data)
y_train = np.array(labels)

print(X_train.shape)   # (number_of_images, 64*64*3)
print(y_train.shape)

(8005, 12288)
(8005,)


## Test Dataset

In [8]:
test_dogs_path = r"C:\Users\KIIT0001\Downloads\Test_dataset_C&D\dogs"
test_cats_path = r"C:\Users\KIIT0001\Downloads\Test_dataset_C&D\cats"

In [9]:
X_test = []
y_test = []

In [10]:
for img_name in os.listdir(test_dogs_path):
    img_path = os.path.join(test_dogs_path, img_name)

    img = cv2.imread(img_path)

    if img is None:
        print("Skipping:", img_name)
        continue

    img = cv2.resize(img, (64, 64))   # resize
    img = img.flatten()               # flatten for ML models

    X_test.append(img)
    y_test.append(1)   # dog = 1

Skipping: _DS_Store


In [11]:
for img_name in os.listdir(test_cats_path):
    img_path = os.path.join(test_cats_path, img_name)

    img = cv2.imread(img_path)

    if img is None:
        print("Skipping:", img_name)
        continue

    img = cv2.resize(img, (64, 64))
    img = img.flatten()

    X_test.append(img)
    y_test.append(0)   # cat = 0

Skipping: _DS_Store


In [12]:
X_test = np.array(X_test)
y_test = np.array(y_test)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (2023, 12288)
y_test shape: (2023,)


# Data Preprocessing

In [13]:
from sklearn.preprocessing import StandardScaler

In [14]:
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [15]:
lr_scaler = StandardScaler()

In [16]:
X_train_lr = lr_scaler.fit_transform(X_train)

In [17]:
X_test_lr = lr_scaler.transform(X_test)

In [18]:
print("X_train_lr shape:", X_train_lr.shape)
print("X_test_lr shape:", X_test_lr.shape)

X_train_lr shape: (8005, 12288)
X_test_lr shape: (2023, 12288)


# Model Hyperparameter Tunning

In [19]:
def extract_hog_features(X):
    hog_features = []
    
    for img in X:
        img = img.reshape(64, 64, 3).astype('uint8')
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm='L2-Hys'
        )
        
        hog_features.append(features)
    
    return np.array(hog_features)

In [20]:
X_train_hog = extract_hog_features(X_train)
X_test_hog  = extract_hog_features(X_test)

print(X_train_hog.shape)
print(X_test_hog.shape)

(8005, 1764)
(2023, 1764)


In [21]:
from sklearn.preprocessing import StandardScaler

lr_scaler = StandardScaler()
X_train_hog = lr_scaler.fit_transform(X_train_hog)
X_test_hog  = lr_scaler.transform(X_test_hog)

# Logistic_Regression_Model_Training_and Prediction

In [22]:
from sklearn.linear_model import LogisticRegression

lr_hog_model = LogisticRegression(
    max_iter=1000,
    C=1.0,
    solver='lbfgs'
)

lr_hog_model.fit(X_train_hog, y_train)

LogisticRegression(max_iter=1000)

In [23]:
from sklearn.metrics import accuracy_score

y_pred_lr_hog = lr_hog_model.predict(X_test_hog)
lr_hog_accuracy = accuracy_score(y_test, y_pred_lr_hog)

print("Logistic Regression Accuracy (HOG):", lr_hog_accuracy)


Logistic Regression Accuracy (HOG): 0.6742461690558577


In [23]:
from sklearn.metrics import accuracy_score

lr_accuracy = accuracy_score(y_test, y_pred_lr)
print("Logistic Regression Accuracy:", lr_accuracy)

Logistic Regression Accuracy: 0.5546218487394958


# Model save

In [24]:
# same hog.pkl will be used togetherwith
import pickle
with open("lr_hog_model.pkl", "wb") as file:
    pickle.dump(lr_hog_model, file)